# L5B15 Email/Outbound — Modeling

Alternative modeling approach on the Email/Outbound/Luggage/Sales L5B15 scoring events.

Reads from `data/processed/luggage_email_outbound.parquet` — run `02_data_ingestion.ipynb` first.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from my_project.features import EXCLUDE_COLS, PEGA_FEATURES, TARGET

print(f"Imports OK — {len(PEGA_FEATURES)} Pega features configured")

In [ ]:
PROCESSED_FILE = Path("../data/processed/luggage_email_outbound.parquet")
df = pd.read_parquet(PROCESSED_FILE)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"Model versions: {df['modelVersion'].nunique()}  ({df['modelVersion'].value_counts().to_dict()})")
df.head(3)

## Feature preparation

In [ ]:
# ── Check configured features ────────────────────────────────────────────
if PEGA_FEATURES:
    available   = [c for c in PEGA_FEATURES if c in df.columns]
    not_in_data = [c for c in PEGA_FEATURES if c not in df.columns]
    print(f"Configured: {len(PEGA_FEATURES)}  |  Found in data: {len(available)}  |  Missing: {len(not_in_data)}")
    if not_in_data:
        print("  Not found in dataset:")
        for c in not_in_data:
            print(f"    - {c}")
    X = df[available].copy()
else:
    # Fallback: use all non-excluded columns when PEGA_FEATURES is empty
    print("PEGA_FEATURES is empty — using all non-excluded columns as fallback")
    X = df.drop(columns=[c for c in EXCLUDE_COLS + [TARGET] if c in df.columns]).copy()
    # Drop near-empty and constant columns
    X = X.loc[:, X.isna().mean() < 0.95]
    X = X.loc[:, X.nunique(dropna=True) > 1]

y = df[TARGET].astype(float)
print(f"X: {X.shape}  |  y: {len(y)} rows")
X.head(3)

## Train / test split

> Consider whether a random split is appropriate.
> With multiple  values a time-based split (earlier → train, later → test)
> better reflects production. The  column can serve as a proxy for time.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

## Model

In [ ]:
# TODO: define and train your model here
# model = ...
# model.fit(X_train, y_train)
# pred = model.predict(X_test)

## Evaluation

In [ ]:
# Uncomment after training:
# print("RMSE: ", np.sqrt(mean_squared_error(y_test, pred)))
# print("MAE:  ", mean_absolute_error(y_test, pred))
# print("R²:   ", r2_score(y_test, pred))
# rho, _ = spearmanr(y_test, pred)
# print("Spearman ρ:", rho)

In [ ]:
# Actual vs predicted
# plt.scatter(y_test, pred, alpha=0.2)
# plt.xlabel("Actual propensity")
# plt.ylabel("Predicted propensity")
# plt.title("Actual vs predicted")
# plt.show()